# Markdown Notes MCP server

This notebook deploys the custom Markdown Notes MCP server and uses its tools to search, save, and read Markdown notes stored on a PVC. See the [README](./README.md) for how the server works and how to build your own image.

MCP servers must be enabled on your prokube cluster (v1.9.0 or later).

## Setup

Load this repository's helpers from [mcp.py](../../src/pk_helpers/mcp.py) and [api_key.py](../../src/pk_helpers/api_key.py). `%run -n` loads their functions and classes into the notebook namespace without running guarded CLI code. `McpSession` is a minimal MCP client; any MCP client library can use the same URLs.

In [ ]:
# mcp.py supplies workspace_mcp_url, server_mcp_url, connect_when_ready and McpSession.
%run -n ../../src/pk_helpers/mcp.py
# api_key.py supplies get_or_create_mcp_api_key, which reads MCP_API_KEY or prompts.
%run -n ../../src/pk_helpers/api_key.py

## Deploy

In [ ]:
!kubectl apply -f markdown-notes.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/markdown-notes-example --timeout=3m

## Connect from this Lab

Labs use the in-cluster Agent Gateway host to reach `/_platform/mcp/<workspace>` without an API key. This workspace federation combines all MCP servers and prefixes each tool with its server name, for example `markdown-notes-example_search_notes`.

In [ ]:
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

internal_url = workspace_mcp_url(namespace)
print(internal_url)

In [ ]:
session = connect_when_ready(internal_url, tool_prefix="markdown-notes-example_")
session.list_tools()

Search the two sample notes the server creates on first start:

In [ ]:
result = session.call_tool("markdown-notes-example_search_notes", {"query": "quota"})
result["structuredContent"]["result"]

Save a note and read it back. Note names are stored as lowercase slugs:

In [ ]:
session.call_tool(
    "markdown-notes-example_save_note",
    {"name": "My first note", "content": "# My first note\n\nSaved from a notebook."},
)
result = session.call_tool("markdown-notes-example_get_note", {"name": "my-first-note"})
print(result["content"][0]["text"])

List all notes:

In [ ]:
result = session.call_tool("markdown-notes-example_list_notes", {})
result["structuredContent"]["result"]

## Connect with an API key

Clients outside the cluster use the server's own route, `https://<your-prokube-domain>/svc/mcp/<workspace>/markdown-notes-example`, with an API key. Create a key with access to `markdown-notes-example` on the [API Keys page](/pkui/ai-gateway/keys) (`https://<your-prokube-domain>/pkui/ai-gateway/keys`), then paste it when prompted below.

The notebook uses the in-cluster host to exercise this same single-server `/svc/mcp` authentication route. Preview its URL before retrieving the key and connecting. Tool names on this route are unprefixed.

In [ ]:
api_key_url = server_mcp_url(namespace, "markdown-notes-example")
print(api_key_url)

In [ ]:
api_key = get_or_create_mcp_api_key()
server_session = connect_when_ready(api_key_url, api_key=api_key)
server_session.list_tools()

## Clean up

This also deletes the PVC with all notes.

In [ ]:
!kubectl delete -f markdown-notes.yaml